In [1]:
import os
import re
import fnmatch
import glob
import sys
module_path = os.path.abspath(os.path.join('./AFIM/src/python'))
sys.path.append(module_path)
import afim
import cmocean
import xesmf             as xe
import numpy             as np
import pandas            as pd
import xarray            as xr
import cosima_cookbook   as cc
import matplotlib.pyplot as plt
import cartopy.crs       as ccrs
import cartopy.feature   as cft
import matplotlib.path      as mpath
from datetime         import datetime,timedelta
import importlib
%matplotlib inline

ModuleNotFoundError: No module named 'pygmt'

In [ ]:
importlib.reload(afim)

In [2]:
cice_prep = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'AFIM','src','python', 'afim_on_gadi.json'))

# JRA55do

In [ ]:
JRA = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/8XDAILY/JRA55_03hr_forcing_tx1_2005.nc')
JRA.spchmd.isel(time=0).plot(figsize=(15,10))

In [ ]:
prra = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_prra_2005.nc')
prsn = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_prsn_2005.nc')
rlds = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_rlds_2005.nc')
rsds = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_rsds_2005.nc')
tas = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_tas_2005.nc')
uas = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_uas_2005.nc')
vas = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_vas_2005.nc')
#psl = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_psl_2005.nc')

In [ ]:
huss = xr.open_dataset('/home/581/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_huss_2005_bilin.nc')
huss.isel(time=0).huss.plot(figsize=(15,10))
G_CICE    = xr.open_dataset('/g/data/jk72/da1339/grids/aom2_cice_0p25_deg.nc')
G_JRA55   = xr.open_dataset('/g/data/jk72/da1339/grids/JRA55do_grid_with_1yr_timeDim.nc')
F_weights = '/g/data/jk72/da1339/grids/weights/rmp_jrar_to_cict_BILINEAR.nc'
huss      = xr.open_dataset('/g/data/qv56/replicas/input4MIPs/CMIP6/OMIP/MRI/MRI-JRA55-do-1-5-0/atmos/3hrPt/huss/gr/v20200916/huss_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_200501010000-200512312100.nc')
rg        = xe.Regridder(G_JRA55, G_CICE, method='bilinear',  periodic=True, filename=F_weights, reuse_weights=True)
HUSS      = rg(huss)
HUSS.huss.isel(time=0).plot(figsize=(15,10))

# Initial Conditions

## re-grid NCAR-gx1 initial conditions to 1/10-degree (or 1/4-degree) AOM2 grid
The first cell below should only need to be run once and then the next cell can be re-run to load and view the contents of the gridded data

In [ ]:
!tar -xvf /scratch/jk72/da1339/afim_input/CESM/CICE_data_gx1_grid_ic-20200320.tar.gz -C /scratch/jk72/da1339/afim_input/CESM/

In [ ]:
cice_prep           = afim.cice_prep(os.path.join(os.path.expanduser('~'),'AFIM', 'src','python', 'afim_on_gadi.json'))
G_CICE              = cice_prep.cice_grid_prep()
G_CESM              = xr.open_dataset('/scratch/jk72/da1339/afim_input/CESM/CICE_data/grid/gx1/global_gx1.bathy.nc')
IC_CESM             = xr.open_dataset('/scratch/jk72/da1339/afim_input/CESM/CICE_data/ic/gx1/iced_gx1_v5.nc')
G_CESM['longitude'] = G_CESM.TLON.values[:,0]
G_CESM['latitude']  = G_CESM.TLAT.values[0,:]
F_weights           = '/g/data/jk72/da1339/grids/weights/map_NCAR-IC-1p0_AOM2-CICE_0p25_bilinear.nc'
rg                  = xe.Regridder(G_CESM, G_CICE, method='bilinear')
IC_CESM             = rg(IC_CESM)
IC_CESM.to_netcdf('/scratch/jk72/da1339/afim_input/CESM/IC_CESM_reG_to_AOM2_op25_2005-01-01.nc')

In [ ]:
IC_CESM

## add aditional fields that are required by CICE6 (as opposed to CICE5)
The additional fields can be seen if one 'looks' at the variables of 
This should be for the date of 01 Jan 2005 and make adjustments to coordinates and variable names to match the example initial conditions from NCAR

In [ ]:
IC_AOM2_OCN = xr.open_dataset("/g/data/cj50/access-om2/raw-output/access-om2-025/025deg_jra55v13_ryf9091_gmredi6/output052/ocean/ocean_month.nc")
IC_AOM2_OCN = IC_AOM2_OCN.isel(time=11).drop('time')
IC_AOM2_I2O = xr.open_dataset("/g/data/cj50/access-om2/raw-output/access-om2-025/025deg_jra55v13_ryf9091_gmredi6/output052/ocean/o2i.nc", decode_times=False)
IC_AOM2_I2O = IC_AOM2_I2O.isel(time=0).drop('time')
IC_AOM2_ICE = xr.open_dataset("/g/data/cj50/access-om2/raw-output/access-om2-025/025deg_jra55v13_ryf9091_gmredi6/output052/ice/OUTPUT/iceh.2004-12.nc")
IC_AOM2_ICE = IC_AOM2_ICE.isel(time=0).drop('time')

In [ ]:
IC_AOM2_OCN

In [ ]:
IC_AOM2_I2O

In [ ]:
IC_AOM2_ICE

### loop through each ocean variable and plot

In [ ]:
IC_AOM2_OCN      = xr.open_dataset("/g/data/cj50/access-om2/raw-output/access-om2-025/025deg_jra55v13_ryf9091_gmredi6/output052/ocean/ocean_month.nc")
D_graph_base     = "/g/data/jk72/da1339/GRAPHICAL/initial_conditions/2004_12/"
IC_AOM2_OCN_vars = {"sea_level"             : {"long_name" : "effective sea level (eta_t + patm/(rho0*g)) on T cells",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "m",
                                               "vmin"      : -3,
                                               "vmax"      : 0,
                                               "cmap"      : cmocean.cm.amp}, 
                    "eta_t"                 : {"long_name" : "surface height on T cells",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "m",
                                               "vmin"      : -3,
                                               "vmax"      : 0,
                                               "cmap"      : cmocean.cm.amp}, 
                    "sea_levelsq"           : {"long_name" : "square of effective sea level (eta_t + patm/(rho0*g)) on T cells",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "m^2",
                                               "vmin"      : 0,
                                               "vmax"      : 6,
                                               "cmap"      : cmocean.cm.amp}, 
                    "mld"                   : {"long_name" : "mixed layer depth determined by density criteria",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "m",
                                               "vmin"      : 0,
                                               "vmax"      : 200,
                                               "cmap"      : cmocean.cm.deep}, 
                    "surface_temp"          : {"long_name" : "Conservative temperature",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "C",
                                               "vmin"      : -3,
                                               "vmax"      : 10,
                                               "cmap"      : cmocean.cm.thermal}, 
                    "surface_salt"          : {"long_name" : "Practical Salinity",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "psu",
                                               "vmin"      : 34,
                                               "vmax"      : 36,
                                               "cmap"      : cmocean.cm.haline}, 
                    "evap"                  : {"long_name" : "mass flux from evaporation/condensation (>0 enters ocean)",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "(kg/m^3)*(m/sec)",
                                               "vmin"      : -0.0002,
                                               "vmax"      : 0,
                                               "cmap"      : cmocean.cm.amp_r}, 
                    "melt"                  : {"long_name" : "water flux transferred with sea ice form/melt (>0 enters ocean)",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "(kg/m^3)*(m/sec)",
                                               "vmin"      : 0,
                                               "vmax"      : 0.001,
                                               "cmap"      : cmocean.cm.amp}, 
                    "sfc_salt_flux_restore" : {"long_name" : "sfc_salt_flux_restore: flux from restoring term",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "kg/(m^2*sec)",
                                               "vmin"      : -6e-7,
                                               "vmax"      : 6e-7,
                                               "cmap"      : cmocean.cm.balance},
                    "sfc_salt_flux_ice"     : {"long_name" : "sfc_salt_flux_ice",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "kg/(m^2*sec)",
                                               "vmin"      : -8e-6,
                                               "vmax"      : 8e-6,
                                              "cmap"      : cmocean.cm.balance},
                    "sfc_salt_flux_coupler" : {"long_name" : "sfc_salt_flux_coupler: flux from the coupler",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "kg/(m^2*sec)",
                                               "vmin"      : -8e-6,
                                               "vmax"      : 8e-6,
                                              "cmap"      : cmocean.cm.balance}, 
                    "net_sfc_heating"       : {"long_name" : "surface ocean heat flux coming through coupler and mass transfer",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "W/m^2",
                                               "vmin"      : 10,
                                               "vmax"      : 1000,
                                               "cmap"      : cmocean.cm.amp},
                    "tau_x"                 : {"long_name" : "i-directed wind stress forcing u-velocity",
                                               "lon_name"  : "xu_ocean",
                                               "lat_name"  : "yu_ocean",
                                               "units"     : "N/m^2",
                                               "vmin"      : -0.4,
                                               "vmax"      :  0.4,
                                               "cmap"      : cmocean.cm.balance},
                    "tau_y"                 : {"long_name" : "j-directed wind stress forcing v-velocity",
                                               "lon_name"  : "xu_ocean",
                                               "lat_name"  : "yu_ocean",
                                               "units"     : "N/m^2",
                                               "vmin"      : -0.4,
                                               "vmax"      :  0.4,
                                               "cmap"      : cmocean.cm.balance},
                    "bmf_u"                 : {"long_name" : "Bottom u-stress via bottom drag",
                                               "lon_name"  : "xu_ocean",
                                               "lat_name"  : "yu_ocean",
                                               "units"     : "N/m^2",
                                               "vmin"      : -0.5,
                                               "vmax"      :  0.5,
                                               "cmap"      : cmocean.cm.balance},
                    "bmf_v"                 : {"long_name" : "Bottom v-stress via bottom drag",
                                               "lon_name"  : "xu_ocean",
                                               "lat_name"  : "yu_ocean",
                                               "units"     : "N/m^2",
                                               "vmin"      : -0.5,
                                               "vmax"      :  0.5,
                                               "cmap"      : cmocean.cm.balance},
                    "tx_trans_int_z"        : {"long_name" : "T-cell i-mass transport vertically summed",
                                               "lon_name"  : "xu_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "kg/s",
                                               "vmin"      : -3e10,
                                               "vmax"      :  3e10,
                                               "cmap"      : cmocean.cm.balance},
                    "ty_trans_int_z"        : {"long_name" : "T-cell j-mass transport vertically summed",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yu_ocean",
                                               "units"     : "kg/s",
                                               "vmin"      : -3e10,
                                               "vmax"      :  3e10,
                                               "cmap"      : cmocean.cm.balance},
                    "aredi"                 : {"long_name" : "neutral diffusivity at k=1",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "m^2/sec",
                                               "vmin"      : 0,
                                               "vmax"      : 200,
                                               "cmap"      : cmocean.cm.amp},
                    "agm"                   : {"long_name" : "GM diffusivity at surface",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "m^2/sec",
                                               "vmin"      : 0,
                                               "vmax"      : 200,
                                               "cmap"      : cmocean.cm.amp},
                    "frazil_3d"             : {"long_name" : "ocn frazil heat flux over time step",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "W/m^2",
                                               "vmin"      : -200,
                                               "vmax"      :  200,
                                               "cmap"      : cmocean.cm.balance},
                    "swflx"                 : {"long_name" : "shortwave flux into ocean (>0 heats ocean)",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "W/m^2",
                                               "vmin"      : 0,
                                               "vmax"      : 350,
                                               "cmap"      : cmocean.cm.amp},
                    "sw_heat"               : {"long_name" : "penetrative shortwave heating",
                                               "lon_name"  : "xt_ocean",
                                               "lat_name"  : "yt_ocean",
                                               "units"     : "W/m^2",
                                               "vmin"      : -150,
                                               "vmax"      : 0,
                                               "cmap"      : cmocean.cm.amp_r}}
for var_name, attributes in IC_AOM2_OCN_vars.items():
    if var_name=="frazil_3d" or var_name=="sw_heat": 
        da = IC_AOM2_OCN[var_name].isel(time=11,st_ocean=0)
    elif var_name=="surface_temp":
        da = da = IC_AOM2_OCN[var_name].isel(time=11)-273.15
    else:
        da = IC_AOM2_OCN[var_name].isel(time=11)
    t_str    = da.time.values.astype('datetime64[M]').astype(str)
    tit_str  = f"{attributes['long_name']}"
    F_name   = f"{t_str}_{var_name}_ACCESS-OM2-025_SH.png"
    cbar_lab = f"{attributes['units']}"
    if not os.path.exists(os.path.join(D_graph_base,F_name)):
        print(f"attempting to plot {var_name}")
        afim.plot_cartopy_pcolormesh(da=da, lon_name=attributes['lon_name'],
                                     lat_name=attributes['lat_name'],t_str=t_str,cmap=attributes['cmap'], 
                                     vmin=attributes['vmin'], vmax=attributes['vmax'],cbar_label=cbar_lab,
                                     title=tit_str,D_save=D_graph_base,F_save=F_name)

### loop through each I2O.nc variable and plot 

In [ ]:
IC_AOM2_I2O      = xr.open_dataset("/g/data/cj50/access-om2/raw-output/access-om2-025/025deg_jra55v13_ryf9091_gmredi6/output052/ocean/o2i.nc", decode_times=False)
D_graph_base     = "/g/data/jk72/da1339/GRAPHICAL/initial_conditions/2004_12/"
IC_AOM2_I2O_vars = {"sst_i"    : {"long_name" : "Sea Surface Temperature",
                                  "lon_name"  : "nx",
                                   "lat_name"  : "ny",
                                   "units"     : "C",
                                   "vmin"      : -3,
                                   "vmax"      : 10,
                                   "cmap"      : cmocean.cm.thermal}, 
                    "sss_i"    : {"long_name" : "Sea Surface Salinity",
                                  "lon_name"  : "nx",
                                   "lat_name"  : "ny",
                                   "units"     : "psu",
                                   "vmin"      : 34,
                                   "vmax"      : 36,
                                   "cmap"      : cmocean.cm.haline}, 
                    "ssu_i"    : {"long_name" : "Sea Surface Zonal Flow",
                                  "lon_name"  : "nx",
                                   "lat_name"  : "ny",
                                   "units"     : "m/s",
                                   "vmin"      : -1.5,
                                   "vmax"      : 1.5,
                                   "cmap"      : cmocean.cm.balance}, 
                    "ssv_i"    : {"long_name" : "Sea Surface Meridional Flow",
                                  "lon_name"  : "nx",
                                   "lat_name"  : "ny",
                                   "units"     : "m/s",
                                   "vmin"      : -1.5,
                                   "vmax"      : 1.5,
                                   "cmap"      : cmocean.cm.balance}, 
                    "sslx_i"   : {"long_name" : "Sea Surface Zonal Tilt",
                                  "lon_name"  : "nx",
                                   "lat_name"  : "ny",
                                   "units"     : "m",
                                   "vmin"      : -8e-6,
                                   "vmax"      :  8e-6,
                                   "cmap"      : cmocean.cm.balance}, 
                    "ssly_i"   : {"long_name" : "Sea Surface Meridional Tilt",
                                  "lon_name"  : "nx",
                                   "lat_name"  : "ny",
                                   "units"     : "m",
                                   "vmin"      : -8e-6,
                                   "vmax"      :  8e-6,
                                   "cmap"      : cmocean.cm.balance}, 
                    "pfmice_i" : {"long_name" : "Freeze Melt",
                                  "lon_name"  : "nx",
                                   "lat_name"  : "ny",
                                   "units"     : "[UNKNOWN]",
                                   "vmin"      : -1.5e6,
                                   "vmax"      : -5.0e4,
                                   "cmap"      : cmocean.cm.amp_r}}
for var_name, attributes in IC_AOM2_I2O_vars.items():
    if var_name in ["sst_i","sss_i","pfmice_i"]:
        if var_name=="sst_i":
            da = IC_AOM2_I2O[var_name].isel(time=0)-273.15
        else:
            da = IC_AOM2_I2O[var_name].isel(time=0)
        lon_name = 'xt_ocean'
        lat_name = 'yt_ocean'
    else:
        da = IC_AOM2_I2O[var_name].isel(time=0)
        lon_name = 'xu_ocean'
        lat_name = 'yu_ocean'
    tit_str  = f"{attributes['long_name']}"
    F_name   = f"2004-12_{var_name}_ACCESS-OM2-025_I2O_SH.png"
    cbar_lab = f"{attributes['units']}"
    print(f"attempting to plot {var_name}")
    fig, ax        = plt.subplots(1,1, figsize=(15,9), dpi=150, facecolor="w", 
                                  subplot_kw=dict(projection=ccrs.SouthPolarStereo()))
    theta          = np.linspace(0, 2*np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts          = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle         = mpath.Path(verts * radius + center)
    ax.set_extent([0,360,-90,-50], ccrs.PlateCarree())
    ax.set_boundary(circle, transform = ax.transAxes)
    ax.add_feature(cft.LAND, color='darkgrey')
    ax.add_feature(cft.COASTLINE, linewidth=.5)
    pcm = ax.pcolormesh(IC_AOM2_OCN[lon_name], IC_AOM2_OCN[lat_name], da, cmap=attributes['cmap'],
                        vmin=da.min().values, vmax=da.max().values, transform=ccrs.PlateCarree())
    plt.colorbar(pcm, label=cbar_lab)
    plt.title(tit_str)
    ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=1, color='gray', alpha=0.25, linestyle='--')
    if not os.path.exists(D_graph_base): os.makedirs(D_graph_base)
    plt.savefig(os.path.join(D_graph_base,F_name))
    plt.show()
    plt.close()
        

### loop through each ACCESS-OM2 ICE output variable and plot

In [ ]:
def plot_variable(da, long_name, units, lon_name, lat_name, D_graph_base, F_name):
    print(f"attempting to plot: {D_graph_base}/{F_name}")
    fig, ax = plt.subplots(1,1, figsize=(15,9), dpi=150, facecolor="w", 
                            subplot_kw=dict(projection=ccrs.SouthPolarStereo()))
    theta = np.linspace(0, 2*np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_extent([0,360,-90,-50], ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.add_feature(cft.LAND, color='darkgrey')
    ax.add_feature(cft.COASTLINE, linewidth=.5)
    pcm = ax.pcolormesh(IC_AOM2_OCN[lon_name], IC_AOM2_OCN[lat_name], da, 
                        cmap=cmocean.cm.amp, transform=ccrs.PlateCarree())
    plt.colorbar(pcm, label=units)
    plt.title(long_name)
    ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=1, color='gray', alpha=0.25, linestyle='--')
    if not os.path.exists(D_graph_base): os.makedirs(D_graph_base)
    plt.savefig(os.path.join(D_graph_base,F_name))
    plt.show()
    plt.close()

In [ ]:
IC_AOM2_ICE = xr.open_dataset("/g/data/cj50/access-om2/raw-output/access-om2-025/025deg_jra55v13_ryf9091_gmredi6/output052/ice/OUTPUT/iceh.2004-12.nc", decode_times=False)
D_graph_base = "/g/data/jk72/da1339/GRAPHICAL/initial_conditions/2004_12/"
for var_name in IC_AOM2_ICE"].data_vars:
    var = IC_AOM2_ICE[var_name]
    if var.dims in [('time', 'nj', 'ni'), ('time', 'nc', 'nj', 'ni')]:
        long_name     = var.long_name
        units         = var.units
        cell_measures = var.cell_measures
        # Determine which latitude/longitude grid to use
        if "tarea" in cell_measures:
            lon_name, lat_name = 'xt_ocean', 'yt_ocean'
        elif "uarea" in cell_measures:
            lon_name, lat_name = 'xu_ocean', 'yu_ocean'
        # If the variable includes the 'nc' dimension, loop through it
        if 'nc' in var.dims:
            for nc_index in range(len(var['nc'])):
                F_name = f"2004-12_{var_name}_nc{nc_index}_ACCESS-OM2-025_CICE_SH.png"
                da = var.isel(time=0, nc=nc_index)
                if not os.path.exists(os.path.join(D_graph_base,F_name)):
                    plot_variable(da, long_name, units, lon_name, lat_name, D_graph_base, F_name)
        else:
            F_name = f"2004-12_{var_name}_ACCESS-OM2-025_CICE_SH.png"
            da = var.isel(time=0)
            if not os.path.exists(os.path.join(D_graph_base,F_name)):
                plot_variable(da, long_name, units, lon_name, lat_name, D_graph_base, F_name)


## get atmosphere short-wave fields and do relevant computations

In [ ]:
msdwswrf = cice_prep.era5_load_and_regrid(era5_var = 'msdwswrf', yr_str = '2005').resample(time='1D').mean('time')
mtdwswrf = cice_prep.era5_load_and_regrid(era5_var = 'mtdwswrf', yr_str = '2005').resample(time='1D').mean('time')
SW_diff  = afim.compute_diffuse_sfc_em(msdwswrf.msdwswrf,mtdwswrf.mtdwswrf)

## creat explicit initial condition dataset from CESM initial conditions

In [ ]:
#IC_ICE5 = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/runs/afim/restart/iced.2005-01-01-00000.nc')
IC_CESM = xr.open_dataset('/scratch/jk72/da1339/afim_input/CESM/IC_CESM_reG_to_AOM2_op25_2005-01-01.nc')

In [ ]:
#hi: grid cell mean ice thickness (m)
IC_ICE5['hi']          = (("ny", "nx"), IC_AOM2_ICE.hi_m"].data)
#hs: grid cell mean snow thickness (m)
IC_ICE5['hs']          = (("ny", "nx"), IC_AOM2_ICE.hs_m"]"].data)
#congel: basal ice growth (m)
IC_ICE5['congel']      = (("ny", "nx"), IC_AOM2.congel_m"].data)
#frazil: frazil ice growth (m)
IC_ICE5['frazil']      = (("ny", "nx"), IC_AOM2.frazil_m"].data)
#snoice: snow–ice formation (m)
IC_ICE5['snoice']      = (("ny", "nx"), IC_AOM2.snoice_m"].data)
#dvidtt: ice volume tendency due to thermodynamics (m/s)
IC_ICE5['dvidtt']      = (("ny", "nx"), IC_AOM2.dvidtt_m"].data)
#dvidtd: ice volume tendency due to dynamics/transport (m/s)
IC_ICE5['dvidtd']      = (("ny", "nx"), IC_AOM2.dvidtd_m"].data)
IC_ICE5['sst']          = (("ny", "nx"), sst"].data-273)
IC_ICE5['sss']          = (("ny", "nx"), sss"].data)
#frzmlt: freezing/melting potential (W/m^2)
IC_ICE5['frzmlt']       = (("ny", "nx"), IC_AOM2.frzmlt_m"].data)

In [ ]:
IC_ICE5.attrs = {'istep1': 2773952,
 'time': 1483228800.0,
 'time_forc': 0.0,
 'nyr': 2005,
 'month': 1,
 'mday': 1,
 'sec': 0}
IC_ICE5

In [ ]:
IC_ICE5.to_netcdf('/scratch/jk72/da1339/afim_input/initial_conditions/iced.2005-01-01-00000-MODIFIED_CESM_NCAR.nc')

## CONSTRUCT IC NETCDF

In [ ]:
IC_CESM_GX1 = xr.open_dataset("/scratch/jk72/da1339/afim_input/CESM/CICE_data/ic/gx1/iced_gx1_v5.nc")
IC_CESM_GX1

In [ ]:
IC                = xr.Dataset()
# COORDINATES
IC.coords["TLON"] = (("ny", "nx"), IC_AOM2_ICE["TLON"].data)
IC.coords["TLAT"] = (("ny", "nx"), IC_AOM2_ICE["TLAT"].data)
IC.coords["ULON"] = (("ny", "nx"), IC_AOM2_ICE["ULON"].data)
IC.coords["ULAT"] = (("ny", "nx"), IC_AOM2_ICE["ULAT"].data)
IC.coords["NCAT"] = (("nc"), IC_AOM2_ICE["nc"].data)
##############################################################################
# VARIABLES/FIELDS FROM IC_AOM2_I2O
#sst: sea-surface temperature (C)
IC['sst']         = (("ny", "nx"), IC_AOM2_I2O["sst_i"].data-273.15)
IC['sss']         = (("ny", "nx"), IC_AOM2_I2O["sss_i"].data)
IC['sice']        = (("ny", "nx"), IC_AOM2_I2O["sss_i"].data)
#frzmlt: freezing/melting potential (W/m^2)
IC['frzmlt']      = (("ny", "nx"), IC_AOM2_I2O["pfmice_i"].data)
#u(v)vel: x(y)-component of ice velocity
IC['uvel']        = (("ny", "nx"), IC_AOM2_I2O["ssu_i"].data)
IC['vvel']        = (("ny", "nx"), IC_AOM2_I2O["ssv_i"].data)
# ss_tltx(y): sea surface in the x(y) direction m
IC['ss_tltx']     = (("ny", "nx"), IC_AOM2_I2O["sslx_i"].data)
IC['ss_tlty']     = (("ny", "nx"), IC_AOM2_I2O["ssly_i"].data)
##############################################################################
# VARIABLES/FIELDS FROM IC_AOM2_ICE
# GRID FIELDS
#tmask: land/boundary mask, thickness (T-cell)
IC['tmask']       = (("ny", "nx"), IC_AOM2_ICE["tmask"].data)
#blkmask:
IC['blkmask']     = (("ny", "nx"), IC_AOM2_ICE["blkmask"].data)
#tarea: area of T-cell (m^2)
IC['tarea']       = (("ny", "nx"), IC_AOM2_ICE["tarea"].data)
#uarea: area of U-cell (m^2)
IC['uarea']       = (("ny", "nx"), IC_AOM2_ICE["uarea"].data)
#dxt: width of T cell (∆𝑥) through the middle (m)
IC['dxt']         = (("ny", "nx"), IC_AOM2_ICE["dxt"].data)
#dyt: width of T cell (∆y) through the middle (m)
IC['dyt']         = (("ny", "nx"), IC_AOM2_ICE["dyt"].data)
#dxu: width of U cell (∆𝑥) through the middle (m)
IC['dxu']         = (("ny", "nx"), IC_AOM2_ICE["dxu"].data)
#dyu: width of U cell (∆y) through the middle (m)
IC['dyu']         = (("ny", "nx"), IC_AOM2_ICE["dyu"].data)
#HTN: length of northern edge (∆x) of T-cell
IC['HTN']         = (("ny", "nx"), IC_AOM2_ICE["HTN"].data)
#HTE: length of eastern edge (∆𝑦) of T-cell
IC['HTE']         = (("ny", "nx"), IC_AOM2_ICE["HTE"].data)
#ANGLE: for conversions between the POP grid and latitude- longitude grids (radians)
IC['ANGLE']       = (("ny", "nx"), IC_AOM2_ICE["ANGLE"].data)
#ANGLET: ANGLE converted to T-cells
IC['ANGLET']      = (("ny", "nx"), IC_AOM2_ICE["ANGLET"].data)
# DYNAMIC FIELDS
#hi: grid cell mean ice thickness (m)
IC['hi']          = (("ny", "nx"), IC_AOM2_ICE["hi_m"].data)
#hs: grid cell mean snow thickness (m)
IC['hs']          = (("ny", "nx"), IC_AOM2_ICE["hs_m"].data)
# Tsfc: temperature of ice/snow top surface (C)
IC['Tsfc']        = (("ny", "nx"), IC_AOM2_ICE["Tsfc_m"].data)
# aice: total concentration of ice in grid cell
IC['aice']        = (("ny", "nx"), IC_AOM2_ICE["aice_m"].data)
# u(v)atm: wind velocity in the x(y) direction (m/s)
IC['uatm']        = (("ny", "nx"), IC_AOM2_ICE["uatm_m"].data)
IC['vatm']        = (("ny", "nx"), IC_AOM2_ICE["vatm_m"].data)
# fs(l)wdn: W/m^2
IC['fsw']         = (("ny", "nx"), IC_AOM2_ICE["fswdn_m"].data)
IC['flw']         = (("ny", "nx"), IC_AOM2_ICE["flwdn_m"].data)
# snow and rain rates (kg/m^2/s), which is (mm/s)
# inputs are given in cm/day so to mm/s then multiply by a factor of 10/86400 or 1/8640
IC['fsnow']       = (("ny", "nx"), IC_AOM2_ICE["snow_ai_m"].data*(1/8640))
IC['frain']       = (("ny", "nx"), IC_AOM2_ICE["rain_ai_m"].data*(1/8640))
# fswfac: scaling factor to adjust ice quantities for updated data
IC['fswfac']       = (("ny", "nx"), IC_AOM2_ICE["fswfac_m"].data)
# fswabs: total absorbed shortwave radiation (W/m^2)
IC['fswabs']       = (("ny", "nx"), IC_AOM2_ICE["fswabs_ai_m"].data)
# alv(n)df: albedo visible (near infrared) diffuse
IC['alvdf']       = (("ny", "nx"), IC_AOM2_ICE["alvdf_m"].data)
IC['alndf']       = (("ny", "nx"), IC_AOM2_ICE["alidf_m"].data)
# albice: bare ice albedo
IC['albice']       = (("ny", "nx"), IC_AOM2_ICE["albice_m"].data)
# albsno: snow albedo
IC['albsno']       = (("ny", "nx"), IC_AOM2_ICE["albsno_m"].data)
# flat: latent heat flux (W/m^2)
IC['flat']       = (("ny", "nx"), IC_AOM2_ICE["flat_ai_m"].data)
# fsens: sensible heat flux (W/m^2)
IC['fsens']       = (("ny", "nx"), IC_AOM2_ICE["fsens_ai_m"].data)
# flwout: outgoing longwave radiation (W/m^2)
IC['flwout']       = (("ny", "nx"), IC_AOM2_ICE["flwup_ai_m"].data)
# evap: evaporative water flux (kg/m^2/s)
# inputs are given in cm/day so to mm/s then multiply by a factor of 10/86400 or 1/8640
IC['evap']       = (("ny", "nx"), IC_AOM2_ICE["evap_ai_m"].data*(1/8640))
# Tair: air temperature at 10 m (C)
IC['Tair']       = (("ny", "nx"), IC_AOM2_ICE["Tair_m"].data)
# congel: basal ice growth (m)
# given as an average rate (cm/day) for the month, so muliply by a factor ( 30.25 (days) / 100 cm/m )
IC['congel']      = (("ny", "nx"), IC_AOM2_ICE["congel_m"].data*(30.25/100))
# frazil: frazil ice growth (m)
IC['frazil']      = (("ny", "nx"), IC_AOM2_ICE["frazil_m"].data*(30.25/100))
# snoice: snow–ice formation (m)
IC['snoice']      = (("ny", "nx"), IC_AOM2_ICE["snoice_m"].data*(30.25/100))
# meltt: top ice melt (m)
IC['meltt']       = (("ny", "nx"), IC_AOM2_ICE["meltt_m"].data*(30.25/100))
# meltb: basal ice melt (m)
IC['meltb']       = (("ny", "nx"), IC_AOM2_ICE["meltb_m"].data*(30.25/100))
# melts: snow melt (m)
IC['melts']       = (("ny", "nx"), IC_AOM2_ICE["melts_m"].data*(30.25/100))
# meltl: lateral melt (m)
IC['meltl']       = (("ny", "nx"), IC_AOM2_ICE["meltl_m"].data*(30.25/100))
# fresh: fresh water flux to ocean (kg/m^2/s)
# inputs are given in cm/day so to mm/s then multiply by a factor of 10/86400 or 1/8640
IC['fresh']       = (("ny", "nx"), IC_AOM2_ICE["fresh_ai_m"].data*(1/8640))
# fhocn: net heat flux to ocean (W/m^2)
IC['fhocn']       = (("ny", "nx"), IC_AOM2_ICE["fhocn_ai_m"].data)
# fswthru: shortwave penetrating to ocean (W/m^2)
IC['fswthru']       = (("ny", "nx"), IC_AOM2_ICE["fswthru_ai_m"].data)
# strairx(y)U: stress on ice by air in the x(y)-direction (centered in U cell) (N/m^2)
IC['strairxU']       = (("ny", "nx"), IC_AOM2_ICE["strairx_m"].data)
IC['strairyU']       = (("ny", "nx"), IC_AOM2_ICE["strairy_m"].data)
# strtltx(y)U: surface stress due to sea surface slope (N/m^2)
IC['strtltxU']       = (("ny", "nx"), IC_AOM2_ICE["strtltx_m"].data)
IC['strtltyU']       = (("ny", "nx"), IC_AOM2_ICE["strtlty_m"].data)
# strocnx(y)U: ice–ocean stress in the x(y)-direction (U-cell) (N/m^2)
IC['strocnxU']       = (("ny", "nx"), IC_AOM2_ICE["strocnx_m"].data)
IC['strocnyU']       = (("ny", "nx"), IC_AOM2_ICE["strocny_m"].data)
# strintx(y)U: divergence of internal ice stress, x(y) (N/m^2)
IC['strintxU']       = (("ny", "nx"), IC_AOM2_ICE["strintx_m"].data)
IC['strintyU']       = (("ny", "nx"), IC_AOM2_ICE["strinty_m"].data)
# strength: ice strength (N/m)
IC['strength']       = (("ny", "nx"), IC_AOM2_ICE["strength_m"].data)
# divu: strain rate I component, velocity divergence (1/s)
# given as 1/day and so need to factor by (1/86400)
IC['divu']       = (("ny", "nx"), IC_AOM2_ICE["divu_m"].data*(1/86400))
# shear: strain rate II component (1/s)
# given as 1/day and so need to factor by (1/86400)
IC['shear']       = (("ny", "nx"), IC_AOM2_ICE["shear_m"].data*(1/86400))
# sig1(2): principal stress components ,  (diagnostic)
IC['sig1']       = (("ny", "nx"), IC_AOM2_ICE["sig1_m"].data)
IC['sig2']       = (("ny", "nx"), IC_AOM2_ICE["sig2_m"].data)
# dvidtt: ice volume tendency due to thermodynamics (m/s)
# given cm/day so multiply by 1.15741e-7
IC['dvidtt']      = (("ny", "nx"), IC_AOM2_ICE["dvidtt_m"].data*(1.15741e-7))
# dvidtd: ice volume tendency due to dynamics/transport (m/s)
# given cm/day so multiply by 1.15741e-7
IC['dvidtd']      = (("ny", "nx"), IC_AOM2_ICE["dvidtd_m"].data*(1.15741e-7))
# daidtt ice area tendency due to thermodynamics (1/s)
# given as 1/day and so need to factor by (1/86400)
IC['daidtt']       = (("ny", "nx"), IC_AOM2_ICE["daidtt_m"].data*(1/86400))
# daidtd: ice area tendency due to dynamics/transport (1/s)
# given as 1/day and so need to factor by (1/86400)
IC['daidtd']       = (("ny", "nx"), IC_AOM2_ICE["daidtd_m"].data*(1/86400))
# mlt_onset: day of year that surface melt begins 
IC['mlt_onset']       = (("ny", "nx"), IC_AOM2_ICE["mlt_onset_m"].data)
# frz_onset: day of year that freezing begins
IC['frz_onset']       = (("ny", "nx"), IC_AOM2_ICE["frz_onset_m"].data)
# fcondtop(n)(_f): conductive heat flux (W/m)
IC['fcondtop']       = (("ny", "nx"), IC_AOM2_ICE["fcondtop_ai_m"].data)
# opening: rate of ice opening due to divergence and shear (1/s)
# given as 1/day and so need to factor by (1/86400)
IC['opening']       = (("ny", "nx"), IC_AOM2_ICE["opening_m"].data*(1/86400))
# # ICE THICKNESS CATEGORIES
# aice(n): total concentration of ice in grid cell (in category n)
IC['aicen']       = (("nc","ny", "nx"), IC_AOM2_ICE["aicen_m"].data)
# vice(n): volume per unit area of ice (in category n) (m)
IC['vicen']       = (("nc","ny", "nx"), IC_AOM2_ICE["vicen_m"].data)
# fsurf(n)(_f): net surface heat flux excluding fcondtop (W/m)
IC['fsurfn']       = (("nc","ny", "nx"), IC_AOM2_ICE["fsurfn_ai_m"].data)
# fcondtop(n)(_f): conductive heat flux (W/m)
IC['fcondtopn']       = (("nc","ny", "nx"), IC_AOM2_ICE["fcondtopn_ai_m"].data)
##############################################################################
# VARIABLES/FIELDS FROM IC_CESM
#scale_factor: scaling factor for shortwave radiation components
# IC['scale_factor'] = (("ny", "nx"), IC_CESM["scale_factor"].data)
# #swv(i)dr(f): incoming shortwave radiation, visible (near IR), direct (diffuse)
# IC['swvdr']        = (("ny", "nx"), IC_CESM["swvdr"].data)
# IC['swvdf']        = (("ny", "nx"), IC_CESM["swvdf"].data)
# IC['swidr']        = (("ny", "nx"), IC_CESM["swidr"].data)
# IC['swidf']        = (("ny", "nx"), IC_CESM["swidf"].data)
# #strocnx(y): ice–ocean stress in the x(y)-direction (U-cell)
# IC['strocnxT']     = (("ny", "nx"), IC_CESM["strocnxT"].data)
# IC['strocnyT']     = (("ny", "nx"), IC_CESM["strocnyT"].data)
# #stressp(1-4): internal ice stress, 𝜎11 + 𝜎22
# IC['stressp_1']    = (("ny", "nx"), IC_CESM["stressp_1"].data)
# IC['stressp_2']    = (("ny", "nx"), IC_CESM["stressp_2"].data)
# IC['stressp_3']    = (("ny", "nx"), IC_CESM["stressp_3"].data)
# IC['stressp_4']    = (("ny", "nx"), IC_CESM["stressp_4"].data)
# #stressm(1-4): internal ice stress, 𝜎11 − 𝜎22
# IC['stressm_1']    = (("ny", "nx"), IC_CESM["stressm_1"].data)
# IC['stressm_2']    = (("ny", "nx"), IC_CESM["stressm_2"].data)
# IC['stressm_3']    = (("ny", "nx"), IC_CESM["stressm_3"].data)
# IC['stressm_4']    = (("ny", "nx"), IC_CESM["stressm_4"].data)
# #stress12(1-4): internal ice stress, 𝜎12
# IC['stress12_1']    = (("ny", "nx"), IC_CESM["stress12_1"].data)
# IC['stress12_2']    = (("ny", "nx"), IC_CESM["stress12_2"].data)
# IC['stress12_3']    = (("ny", "nx"), IC_CESM["stress12_3"].data)
# IC['stress12_4']    = (("ny", "nx"), IC_CESM["stress12_4"].data)
# #iceumask: ice extent mask (U-cell)
# IC['iceumask']      = (("ny", "nx"), IC_CESM["iceumask"].data)
# #vso(n): volume per unit area of snow (in category n) (m)
# IC['vsnon']         = (("ncat","ny", "nx"), IC_CESM["vsnon"].data)
# #Tsfc(n): temperature of ice/snow top surface (in category n) (C)
# IC['Tsfcn']         = (("ncat","ny", "nx"), IC_CESM["Tsfcn"].data)
# #iage:
# IC['iage']          = (("ncat","ny", "nx"), IC_CESM["iage"].data)
# #FY:
# IC['FY']            = (("ncat","ny", "nx"), IC_CESM["FY"].data)
# #alvl:
# IC['alvl']          = (("ncat","ny", "nx"), IC_CESM["alvl"].data)
# #vlvl
# IC['vlvl']          = (("ncat","ny", "nx"), IC_CESM["vlvl"].data)
# #apnd: area concentration of melt ponds
# IC['apnd']          = (("ncat","ny", "nx"), IC_CESM["apnd"].data)
# #hpnd:
# IC['hpnd']          = (("ncat","ny", "nx"), IC_CESM["hpnd"].data)
# #ipnd
# IC['ipnd']          = (("ncat","ny", "nx"), IC_CESM["ipnd"].data)
# #dhs: depth difference for snow on sea ice and pond ice
# IC['dhs']           = (("ncat","ny", "nx"), IC_CESM["dhs"].data)
# #ffrac: fraction of fsurfn used to melt pond ice
# IC['ffrac']         = (("ncat","ny", "nx"), IC_CESM["ffrac"].data)
# #fbrn:
# IC['fbrn']          = (("ncat","ny", "nx"), IC_CESM["fbrn"].data)
# #first_ice: flag for initial ice formation
# IC['first_ice']     = (("ncat","ny", "nx"), IC_CESM["first_ice"].data)
# #sice(001-7):
# IC['sice001']       = (("ncat","ny", "nx"), IC_CESM["sice001"].data)
# IC['sice002']       = (("ncat","ny", "nx"), IC_CESM["sice002"].data)
# IC['sice003']       = (("ncat","ny", "nx"), IC_CESM["sice003"].data)
# IC['sice004']       = (("ncat","ny", "nx"), IC_CESM["sice004"].data)
# #qice(001-7)
# IC['qice001']       = (("ncat","ny", "nx"), IC_CESM["qice001"].data)
# IC['qice002']       = (("ncat","ny", "nx"), IC_CESM["qice002"].data)
# IC['qice003']       = (("ncat","ny", "nx"), IC_CESM["qice003"].data)
# IC['qice004']       = (("ncat","ny", "nx"), IC_CESM["qice004"].data)
# #qsno001:
# IC['qsno001']       = (("ncat","ny", "nx"), IC_CESM["qsno001"].data)
##############################################################################
# DATASET ATTRIBUTES
IC.attrs            = {'istep1'    : 2773952,
                       'time'      : 1483228800.0,
                       'time_forc' : 0.0,
                       'nyr'       : 2005,
                       'month'     : 1,
                       'mday'      : 1,
                       'sec'       : 0}

In [ ]:
IC.to_netcdf('/scratch/jk72/da1339/afim_input/initial_conditions/IC_AOM2_CICE6_2005-01-01.nc')

# AOM2 coallation for stand-alone forcing of CICE6

In [ ]:
importlib.reload(afim)

In [ ]:
cice_prep = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'src', 'python', 'afim_on_gadi.json'))
cice_prep.acom2_coallate_for_cice_forcing()

# construct yearly OCN dataset from AC-OM2-01
1. request data using cosima cookbook
2. compute sea surface slope using eta 
3. pull in the qdp ... requires compute_ocean_qdp_from 

In [ ]:
geolon  = xr.open_dataset('/g/data/ik11/grids/ocean_grid_01.nc').geolon_t.swap_dims({'grid_y_T':'nj','grid_x_T':'ni'})
geolat  = xr.open_dataset('/g/data/ik11/grids/ocean_grid_01.nc').geolat_t.swap_dims({'grid_y_T':'nj','grid_x_T':'ni'})

In [ ]:
session = cc"].database.create_session()
eta     = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='sea_level',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
sst     = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='surface_temp',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
sss     = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='surface_salt',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
mld     = xr.full_like(sst,50)#cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='mld',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
u       = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='u',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00').isel(st_ocean=0)
v       = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='v',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00').isel(st_ocean=0)
qdp     = xr.full_like(sst,0)
G_CICE  = xr.open_dataset(cice_prep.F_G_CICE_original)
dhdx    = afim.compute_ocn_sfc_slope(eta, G_CICE.hte"].data, direction='x')
dhdy    = afim.compute_ocn_sfc_slope(eta, G_CICE.htn"].data, direction='y')

In [ ]:
lat_mom = sst.yt_ocean
lon_mom = sst.xt_ocean
G_tmp   = xr.Dataset({'lon':lon_mom,'lat':lat_mom})
F_net   = cice_prep.compute_era5_net_atmospheric_surface_heat_flux(G_dst              = G_tmp,
                                                              yr_str             = str(2005),
                                                              mean_type          = 'monthly',
                                                              regrid_method      = cice_prep.ERA5_regrid_method,
                                                              regrid_periodicity = cice_prep.ERA5_periodicity,
                                                              cnk_dict           = cice_prep.ERA5_chunking,
                                                              generate_weights   = cice_prep.ERA5_regen_weights,
                                                              weights_file_name  = cice_prep.ERA5_regen_weights)
mld_k  = mld.astype(np.single).load()
Fnet_k = F_net.astype(np.single).load()
print("\t\tslicing temporal temperature derivative at the mixed layer depth")
dTdt_k = dTdt.sel(st_ocean=mld_k,method='nearest').drop('st_ocean').astype(np.single).load()
print("\t\tslicing temporal temperature at the mixed layer depth")
temp_k = temp.sel(st_ocean=mld_k,method='nearest').drop('st_ocean').astype(np.single).load()
print("\t\tslicing salinity temperature at the mixed layer depth")
salt_k = salt.sel(st_ocean=mld_k,method='nearest').drop('st_ocean').astype(np.single).load()
print("\t\tcomputing ocean heat capacity at the mixed layer depth")
cp_k = afim.compute_ocn_heat_capacity_at_depth(salt_k, temp_k, mld_k, mld.yt_ocean).astype(np.single)
cp_k = cp_k.load()
print("\t\tcomputing ocean density at the mixed layer depth")
rho_k = afim.compute_ocn_density_at_depth(salt_k, temp_k, mld_k, mld.yt_ocean).astype(np.single)
rho_k = rho_k.load()
print("\t\tcomputing ocean heat flux at the mixed layer depth")
qdp_k = compute_ocn_heat_flux_at_depth(rho_k, cp_k, mld_k, Fnet_k, dTdt_k)
qdp_k = qdp_k.load()
qdp_k = qdp_k.assign_coords(time=temp.time.isel(time=k).values).expand_dims('time').astype(np.single).to_dataset(name='qdp')

In [ ]:
interp_dates = pd.date_range(start='2005-01-01 12:00',periods=365,freq='1D')
u_int    = u.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
v_int    = v.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
mld_int  = mld.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})*0
sss_int  = sss.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
sst_int  = sst.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
dhdx_int = dhdx.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
dhdy_int = dhdy.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
qdp_int  = qdp.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
mld_int  = mld.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})

In [ ]:
data_vars = {'qdp' : (['time','nj','ni'],qdp_int"].data,{'units':'W/m^2','long_name':'deep ocean heat flux','_FillValue':-2e8}),
             'T'   : (['time','nj','ni'],sst_int"].data,{'units':'K','long_name':'sea surface temperature','_FillValue':-2e8}),
             'S'   : (['time','nj','ni'],sss_int"].data,{'units':'psu','long_name':'sea surface salinity','_FillValue':-2e8}),
             'hblt': (['time','nj','ni'],mld_int"].data,{'units':'m','long_name':'mixed layer depth','_FillValue':-2e8}),
             'U'   : (['time','nj','ni'],u_int"].data,{'units':'m/s','long_name':'meridional surface current','_FillValue':-2e8}),
             'V'   : (['time','nj','ni'],v_int"].data,{'units':'m/s','long_name':'zonal surface current','_FillValue':-2e8}),
             'dhdx': (['time','nj','ni'],dhdx_int"].data,{'units':'m','long_name':'zonal sea surface slope','_FillValue':-2e8}),
             'dhdy': (['time','nj','ni'],dhdy_int"].data,{'units':'m','long_name':'meridional sea surface slope','_FillValue':-2e8})}
coords = {'xc'   : (['nj','ni'],geolon"].data,{'units':'degrees_east'}),
          'yc'   : (['nj','ni'],geolat"].data,{'units':'degrees_north'}),
          'time' : (['time'],sst_int.time"].data,{'long_name':'time','cartesian_axis':'T','calendar_type':'GREGORIAN'})}
attrs = {'creation_date': datetime.now().strftime('%Y-%m-%d %H'),
         'conventions'  : "CCSM data model domain description -- for CICE6 standalone 'NCAR' ocean option",
         'title'        : "daily averaged ocean forcing from ACCESS-OM2-01 output for CICE6 standalone ocean forcing",
         'source'       : "ACCESS-OM2-01",
         'comment'      : "source files found on gadi, /g/data/cj50/access-om2/raw-output/access-om2-01/01deg_jra55v140_iaf",
         'calendar'     : "standard",
         'note1'        : "grid is ACCESS-OM2 t-grid",
         'note2'        : "u,v are left un-interpolated to t-grid (i.e. they are from ACCESS-OM2 u-grid)",
         'note3'        : "source files were concatenated using NCO tools, see python afim module method concat_access_om2",
         'note4'        : "dhdx,dhdy are described in ACCESS-OM2 output as effective sea level (eta_t + patm/(rho0*g)) on T cells",
         'note4a'       : "I have computed sea surface slope using python afim module function compute_ocn_sfc_slope, which assumes input is eta_t",
         'author'       : 'Daniel P Atwater',
         'email'        : 'daniel.atwater@utas.edu.au'}
F_OCN     = '/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p1/monthly/acom2_ocn_frcg_cice6_0p1_2005_zeroed_qdp_scalar50m_hblt_scipy_interp.nc'
OCN       = xr.Dataset(data_vars=data_vars,coords=coords,attrs=attrs)
enc_dict  = {'shuffle':True,'zlib':True,'complevel':5}
write_job = OCN.to_netcdf(F_OCN, unlimited_dims=['time'], compute=False,
                          encoding={'qdp':enc_dict, 'T':enc_dict, 'S':enc_dict, 'hblt':enc_dict,
                                    'U':enc_dict, 'V':enc_dict, 'dhdx':enc_dict, 'dhdy':enc_dict})
#write_job = OCN.to_netcdf(F_OCN, unlimited_dims=['time'], compute=True,
#                          encoding={'qdp':enc_dict, 'T':enc_dict, 'S':enc_dict, 'hblt':enc_dict,
#                                    'U':enc_dict, 'V':enc_dict, 'dhdx':enc_dict, 'dhdy':enc_dict})
with ProgressBar():
    print(f"Writing to {F_OCN}")
    write_job.compute()

In [ ]:
sst.isel(time=180).plot(figsize=(20,10))
sss.isel(time=180).plot(figsize=(20,10))
mld.isel(time=180).plot(figsize=(20,10))
u.isel(time=180).plot(figsize=(20,10))
v.isel(time=180).plot(figsize=(20,10))
dhdx.isel(time=180).plot(figsize=(20,10))
dhdy.isel(time=180).plot(figsize=(20,10))

In [ ]:
xr.open_dataset('/scratch/jk72/da1339/qdp/tmp/qdp_yd129.nc').qdp.plot(figsize=(20,10))

In [ ]:
variables = cc.querying.get_variables(session, experiment='01deg_jra55v140_iaf')
list(variables.name)

In [ ]:
IC_gx3 = xr.open_dataset('/g/data/jk72/da1339/cice-dirs/input/CICE_data/ic/gx3/iced_gx3_v6.2005-01-01.nc')
G_gx3 = xr.open_dataset('/g/data/jk72/da1339/cice-dirs/input/CICE_data/grid/gx3/grid_gx3.nc')

In [ ]:
IC_gx3['lat'] = (['nj','ni'],G_gx3.ulat.values*(180/np.pi),{'units':'degrees_north','_FillValue':-2e8})
IC_gx3['lon'] = (['nj','ni'],G_gx3.ulon.values*(180/np.pi),{'units':'degrees_east','_FillValue':-2e8})
IC_gx3['htn'] = (['nj','ni'],G_gx3.htn.values*100,{'units':'m','_FillValue':-2e16})
IC_gx3['hte'] = (['nj','ni'],G_gx3.hte.values*100,{'units':'m','_FillValue':-2e16})
IC_gx3['hus'] = (['nj','ni'],G_gx3.hus.values*100,{'units':'m','_FillValue':-2e16})
IC_gx3['angle'] = (['nj','ni'],G_gx3.angle.values*(180/np.pi),{'units':'degrees','_FillValue':-2e16})

In [ ]:
G_CICE        = xr.open_dataset(cice_prep.F_G_CICE_original)
G_CICE['lat'] = (['ny','nx'],G_CICE.tlat.values*(180/np.pi),{'units':'degrees_north','_FillValue':-2e8})
G_CICE['lon'] = (['ny','nx'],G_CICE.tlon.values*(180/np.pi),{'units':'degrees_east','_FillValue':-2e8})
G_CICE        = G_CICE.drop(('ulat','ulon','tlat','tlon','clon_t','clat_t','clat_u','clon_u','angle','uarea'))
G_CICE        = G_CICE.assign_coords(xt_ocean=G_CICE['lon'][0,:],yt_ocean=G_CICE['lat'][:,0])
G_CICE        = G_CICE.set_index(nx='xt_ocean',ny='yt_ocean')

In [ ]:
rg = xe.Regridder(IC_gx3, G_CICE, method='patch')
IC = rg(IC_gx3)

In [ ]:
IC.aicen.isel(ncat=0).plot(figsize=(20,10))

In [ ]:
G_BRAN         = xr.open_dataset("/g/data/gb6/BRAN/BRAN2020/static/ocean_grid.nc")
LN,LT          = np.meshgrid(G_BRAN.xt_ocean.values,G_BRAN.yt_ocean.values)
G_BRANt['lon'] = (['yt_ocean','xt_ocean'],LN,{'units':'degrees_east'})
G_BRANt['lat'] = (['yt_ocean','xt_ocean'],LT,{'units':'degrees_north'})
G_BRANt        = G_BRANt.drop(('xu_ocean','yu_ocean','hu','kmu','umask','tmask','st_edges_ocean','Time','st_ocean'))
G_BRANt

In [ ]:
rg = xe.Regridder(G_BRAN, G_CICE, 'bilinear', filename='/g/data/jk72/da1339/grids/weights/bran_ac_om2_01_bilinear.nc', reuse_weights=False)

In [ ]:
start_date = cice_prep.regrid_start_date
dt_obj = datetime.strptime(start_date,'%Y-%m-%d').date() + pd.DateOffset(years=0)
spdts  = cice_prep.time_series_day_month_start_or_end(start_date=dt_obj.strftime('%Y-%m-%d'),month_start=False)
stdts  = cice_prep.time_series_day_month_start_or_end(start_date=dt_obj.strftime('%Y-%m-%d'))
yr_str = cice_prep.define_year_string(stdts[0])
G_CICE = cice_prep.cice_grid_prep()
G_BRAN = cice_prep.bran_grid_prep()
rg     = xe.Regridder(G_BRAN, G_CICE, method='bilinear',  periodic=True,  filename=cice_prep.F_BRAN_weights, reuse_weights=True)
temp_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_temp_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
temp   = rg(temp_n)
temp   = temp.drop(('nv'))
salt_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_salt_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
salt   = rg(salt_n)
salt   = salt.drop(('nv'))
mld_n  = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_mld_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
mld    = rg(mld_n)
mld    = mld.drop('nv')
uocn_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_u_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
uocn_n = uocn_n.isel(st_ocean=0)
uocn_n = uocn_n.rename(xu_ocean='xt_ocean',yu_ocean='yt_ocean')
uocn   = rg(uocn_n)  
uocn   = uocn.drop(('st_ocean','nv','st_edges_ocean'))
vocn_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_v_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
vocn_n = vocn_n.isel(st_ocean=0)
vocn_n = vocn_n.rename(xu_ocean='xt_ocean',yu_ocean='yt_ocean')
vocn   = rg(vocn_n)
vocn   = vocn.drop(('st_ocean','nv','st_edges_ocean'))
eta_n  = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_eta_t_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
eta    = rg(eta_n)
eta    = eta.drop(('nv'))
sst    = temp.sel(st_ocean=0,method='nearest')
sss    = salt.sel(st_ocean=0,method='nearest')
dhdx   = afim.compute_ocn_sfc_slope(eta, G_CICE, direction='x', grid_scale_factor=100)
dhdy   = afim.compute_ocn_sfc_slope(eta, G_CICE, direction='y', grid_scale_factor=100)
# compute qdp
print("computing temperature derivative over time")
dTdt    = temp.differentiate("Time")
print("computing atmospheric surface heat flux")
G_ERA5    = cice_prep.era5_grid_prep()
rg        = xe.Regridder(G_ERA5, G_CICE, method='bilinear', periodic=True, filename=cice_prep.F_ERA5_weights, reuse_weights=True)
dt0       = datetime.strptime(start_date,'%Y-%m-%d') + timedelta(days=(0*365))
lw_nat    = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msdwlwrf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
sw_nat    = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msdwswrf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
msshf_nat = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msshf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
mslhf_nat = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'mslhf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
F_net_nat = sw_nat.msdwswrf - lw_nat.msdwlwrf - mslhf_nat.mslhf - msshf_nat.msshf
F_net_rg  = rg(F_net_nat)
F_net_rg  = F_net_rg.assign_coords(lon=G_CICE.lon[0,:],lat=G_CICE.lat[:,0])
F_net_rg  = F_net_rg.rename({'ny':'yt_ocean','nx':'xt_ocean'})
F_net_rg  = F_net_rg.resample(time='1D').mean()
print("computing ocean heat capacity at the mixed layer depth")
cp   = afim.compute_ocn_heat_capacity_at_depth(salt.salt, temp.temp, mld.mld, mld.ny)
#cp   = cp.drop(('st_ocean','nv','st_edges_ocean'))
print("computing ocean density at the mixed layer depth")
rho  = afim.compute_ocn_density_at_depth(salt.salt, temp.temp, mld.mld, mld.ny)
#rho  = rho.drop(('st_ocean','nv','st_edges_ocean'))

In [ ]:
rho  = afim.compute_ocn_density_at_depth(salt.salt, temp.temp, mld.mld, mld.ny)

In [ ]:
cp.isel(st_ocean=0,Time=0).plot(figsize=(20,10))

In [ ]:
rho.isel(st_ocean=0,Time=0).plot(figsize=(20,10))

In [ ]:
start_date         = cice_prep.BRAN_start_date
D_BRAN             = cice_prep.D_BRAN
regrid_method      = cice_prep.BRAN_regrid_method
regrid_periodicity = cice_prep.BRAN_periodicity
var_freq           = cice_prep.BRAN_var_freq
F_weights          = cice_prep.F_BRAN_t_weights
cnk_dict           = cice_prep.BRAN_t_chunking
G_BRAN             = cice_prep.bran_grid_prep()
G_dst              = cice_prep.cice_grid_prep()
dt_obj             = cice_prep.define_datetime_object(start_date=start_date, year_offset=0)
yr_str             = cice_prep.year_string_from_datetime_object(dt_obj)
mo_str             = '{:02d}'.format(0+1)

In [ ]:
fig = plt.figure(figsize=(15,12))
axs = plt.scatter(G_BRAN.lon,G_BRAN.lat)

In [ ]:
fig = plt.figure(figsize=(15,12))
axs = plt.scatter(G_dst.lon,G_dst.lat)

In [ ]:
rg = xe.Regridder(G_BRAN, G_dst, method=regrid_method,  periodic=regrid_periodicity, filename=F_weights, reuse_weights=False)

# construct yearly OCN dataset from BRAN

1. request data using cosima cookbook
2. compute sea surface slope using eta
3. pull in the qdp ... requires compute_ocean_qdp_from

In [ ]:
cice_prep.bran_load_and_regrid( bran_var='temp' , yr_str='2005', mo_str='05', grid_type='t')

# SCRATCH CICE PREPERATION

In [ ]:
session = cc.database.create_session()
pd.set_option("display.max_rows", None)
cc.querying.get_experiments(session)

In [2]:
era5 = xr.open_dataset("/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/ERA5/24XDAILY/era5_for_cice6_2005.nc")
era5

<xarray.Dataset>
Dimensions:  (time: 744, ny: 1080, nx: 1440)
Coordinates:
    LON      (ny, nx) float64 ...
    LAT      (ny, nx) float64 ...
  * time     (time) datetime64[ns] 2005-01-01 ... 2005-01-31T23:00:00
Dimensions without coordinates: ny, nx
Data variables:
    airtmp   (time, ny, nx) float32 ...
    dlwsfc   (time, ny, nx) float32 ...
    glbrad   (time, ny, nx) float32 ...
    spchmd   (time, ny, nx) float32 ...
    ttlpcp   (time, ny, nx) float32 ...
    wndewd   (time, ny, nx) float32 ...
    wndnwd   (time, ny, nx) float32 ...
Attributes:
    creation_date:  2023-10-25 03
    conventions:    CCSM data model domain description -- for CICE6 standalon...
    title:          re-gridded ERA5 for CICE6 standalone atmosphere forcing
    source:         ERA5, https://doi.org/10.1002/qj.3803, 
    comment:        source files found on gadi, /g/data/rt52/era5/single-leve...
    note1:          ERA5 documentation, https://confluence.ecmwf.int/display/...
    note2:          regridding weight file, /g/data/jk72/da1339/grids/weights...
    note3:          re-gridded using ESMF_RegridGenWeights
    author:         Daniel P Atwater
    email:          daniel.atwater@utas.edu.au

In [3]:
era5.airtmp.isel(time=0).min()

<xarray.DataArray 'airtmp' ()>
array(216.24781799)
Coordinates:
    time     datetime64[ns] 2005-01-01

In [ ]:
jra55 = xr.open_dataset("/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/8XDAILY/JRA55_03hr_forcing_tx1_2005.nc")
jra55

In [ ]:
jra55.airtmp.isel(time=0).min()

In [ ]:
t2m = xr.open_dataset("/g/data/rt52/era5/single-levels/reanalysis/2t/2005/2t_era5_oper_sfc_20050101-20050131.nc")
t2m

In [ ]:
t2m.t2m.isel(time=0).min()

In [3]:
yr_str        = '2005'
G_origin      = 'aom2'
G_res         = '0p25'
regrid_method = 'nearest_s2d'
regrid_prdcty = False
F_weights     = "/g/data/jk72/da1339/grids/weights/map_ERA5_{origin:s}_{res:s}_{method:s}.nc".format(origin=G_origin,res=G_res,method=regrid_method)
reuse_weights = False
G_CICE        = xr.open_dataset('/g/data/ik11/inputs/access-om2/input_20200530/cice_025deg/grid.nc')
G_CICE['lat'] = (['ny','nx'],G_CICE.tlat.values*(180/np.pi),{'units':'degrees_north'})
G_CICE['lon'] = (['ny','nx'],G_CICE.tlon.values*(180/np.pi),{'units':'degrees_east'})
print('Loading and regridding 2t')
ERA5_oG       = xr.open_mfdataset( os.path.join('/g/data/rt52/era5/single-levels/reanalysis','2t'     ,yr_str,'*0101-*.nc') , chunks={'time': 10, 'ny': 100, 'nx': 100} )
LON,LAT       = np.meshgrid(ERA5_oG.longitude.values,ERA5_oG.latitude.values)
G_ERA5        = xr.Dataset(data_vars={'lon':(('ny','nx'),LON),
                                      'lat':(('ny','nx'),LAT)})
rg            = xe.Regridder(G_ERA5, G_CICE, method=regrid_method,  periodic=regrid_prdcty, filename=F_weights, reuse_weights=reuse_weights)
t2m_rg           = rg(ERA5_oG)
t2m_rg_long_name = ERA5_oG['t2m'].attrs['long_name']
t2m_rg_units     = ERA5_oG['t2m'].attrs['units']
t2m_rg

Loading and regridding 2t


<xarray.Dataset>
Dimensions:  (time: 744, ny: 1080, nx: 1440)
Coordinates:
  * time     (time) datetime64[ns] 2005-01-01 ... 2005-01-31T23:00:00
Dimensions without coordinates: ny, nx
Data variables:
    t2m      (time, ny, nx) float32 dask.array<chunksize=(10, 1080, 1440), meta=np.ndarray>
Attributes:
    regrid_method:  nearest_s2d

In [4]:
t2m_rg.t2m.isel(time=0).min().values

array(216.24782, dtype=float32)